In [2]:
from typing import TypedDict, List, Optional, Dict, Union, Any
from dotenv import load_dotenv  
from langchain_core.messages import BaseMessage # The foundational class for all message types in LangGraph
from langchain_core.messages import ToolMessage # Passes data back to LLM after it calls a tool such as the content and the tool_call_id
from langchain_core.messages import SystemMessage # Message for providing instructions to the LLM
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
import httpx
import os

In [3]:
load_dotenv()

True

In [4]:
class AgentState(TypedDict):
    """State of the agent, including the chat history and tool calls."""
    chat_history: List[BaseMessage]
    flight_details: Dict[str, Any]                  # Replace with a more specific TypedDict if you have flight schema
    hotel_candidates: List[Dict[str, Any]]          # List of hotel search results
    selected_hotel: Dict[str, Any]                  # Hotel confirmed by user
    tool_output: Any                                # Output from the last tool (can be anything)
    agent_action: Union[str, Dict[str, Any]]        # Tool name or full action object from LLM


In [6]:
@tool
def fetch_flights(state: AgentState, from_id, to_id, adults, children, travel_class, currency, stops="none", page_no="1", sort="BEST") -> AgentState:
    """Fetch flight details based on user input."""
    url = "https://booking-com15.p.rapidapi.com/api/v1/flights/searchFlights"

    headers = {
    "x-rapidapi-key": f"Bearer {os.getenv('x-rapidapi-key')}",
	"x-rapidapi-host": "booking-com15.p.rapidapi.com"
    }
    
    querystring = {
        "fromId": from_id,
        "toId": to_id,
        "stops": stops,
        "pageNo": page_no,
        "adults": str(adults),
        "children": children,  # e.g. "0,17" for two kids aged 0 and 17
        "sort": sort,
        "cabinClass": travel_class.upper(),  # "ECONOMY", "BUSINESS", etc.
        "currency_code": currency.upper()
    }
    try:
        response = httpx.get(url, headers=headers, params=querystring)
        response.raise_for_status()  # Raise an error for bad responses
        raw_flights = response.json().get("data", {}).get("flightOffers", [])

        state["flight_details"] = [
    {
        "airline": f["segments"][0]["legs"][0]["carriersData"][0].get("name"),
        "departure": f["segments"][0]["legs"][0].get("departureTime"),
        "arrival": f["segments"][0]["legs"][0].get("arrivalTime"),
        "duration": f["segments"][0]["legs"][0].get("totalTime", 0) // 60,  # seconds to minutes
        "stops": len(f["segments"][0]["legs"][0].get("flightStops", [])),
        "price": round(
            f["priceBreakdown"]["total"]["units"] + f["priceBreakdown"]["total"]["nanos"] / 1e9, 2
        ),
        "currency": f["priceBreakdown"]["total"].get("currencyCode"),
        "cabin_class": f["segments"][0]["legs"][0].get("cabinClass")
    }
    for f in raw_flights[:10]  # top 10 results
]

    except httpx.RequestError as e:
        print(f"Request error occurred: {e}")
        state["flight_details"] = []

    return state

In [ ]:
@tool
def fetch_hotels(state: AgentState, dest_id: str, search_type: str = "CITY", adults: int = 1, children_age: str = "", room_qty: int = 1, page_number: int = 1, units: str = "metric", temperature_unit: str = "c", languagecode: str = "en-us", currency_code: str = "AED", location: str = "US") -> AgentState:
    """Fetch hotel details based on user input."""
    try:
        url = "https://booking-com15.p.rapidapi.com/api/v1/hotels/searchHotels" 
        querystring = {
            "dest_id": dest_id,
            "search_type": search_type,
            "adults": str(adults),
            "children_age": children_age,
            "room_qty": str(room_qty),
            "page_number": str(page_number),
            "units": units,
            "temperature_unit": temperature_unit,
            "languagecode": languagecode,
            "currency_code": currency_code,
            "location": location
        }
        headers = {
        "x-rapidapi-key": f"Bearer {os.getenv('x-rapidapi-key')}",
        "x-rapidapi-host": "booking-com15.p.rapidapi.com"
        }

        response = httpx.get(url, headers=headers, params=querystring)

        raw_hotels = response.json().get("data", {})

        state["hotel_candidates"] = [
        {
            "dest_id": h.get("dest_id"),
            "dest_type": h.get("dest_type"),
            "name": h.get("name"),
            "label": h.get("label"),
            "city": h.get("city_name"),
            "region": h.get("region"),
            "country": h.get("country"),
            "latitude": h.get("latitude"),
            "longitude": h.get("longitude"),
            "nr_hotels": h.get("nr_hotels"),
            "image_url": h.get("image_url")
        }
        for h in raw_hotels[:10]  # top 10 search results
        ]
    
    except httpx.RequestError as e:
        print(f"Request error occurred: {e}")
        state["hotel_candidates"] = []

    return state
